<a href="https://colab.research.google.com/github/er1cbrown/BiostatsPython/blob/main/Web_ScraperhotEncoded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Base URL for scraping
base_url = "https://www.draftsharks.com/injury-predictor/"
positions = ["qb"]  # Add other positions if needed

# Define an empty DataFrame to store player data
players_data = pd.DataFrame(columns=[
    'Name', 'Seasons Played', 'Height', 'Weight', 'Position',
    'Injury Date', 'Injury Type', 'Injury Details'
])

for position in positions:
    url = f"{base_url}{position}/"
    response = requests.get(url)

    if response.status_code != 200:
        print(f"Failed to retrieve data from {url}. Status code: {response.status_code}")
        continue

    soup = BeautifulSoup(response.content, "html.parser")
    body = soup.find("body")

    if not body:
        print("Failed to retrieve data from body.")
        continue

    content_box = body.find("div", class_="content-box")
    if not content_box:
        print("Failed to retrieve data from content-box.")
        continue

    table_sip_info = content_box.find("div", class_="table-sip-info")
    if not table_sip_info:
        print("Failed to retrieve data from table-sip-info.")
        continue

    sip_table = table_sip_info.find("table", class_="sip-table")
    if not sip_table:
        print("Failed to retrieve data from sip-table.")
        continue

    tbody = sip_table.find("tbody")
    if not tbody:
        print("Failed to retrieve data from tbody.")
        continue

    rows = tbody.find_all("tr", class_="expand-row collapse")

    for row in rows:  # Iterate through each player row
      extra_player_info = row.find("div", class_="extra-player-info")
      if not extra_player_info:
          continue  # Skip if player info is missing

      t_row = extra_player_info.find("div", class_="row")
      if not t_row:
          continue

      player_stats = t_row.find("div", class_="player-stats")
      if not player_stats:
          continue

      stat_row = player_stats.find("div", class_="row")
      if not stat_row:
          continue

      col_lg_three = stat_row.find("div", class_="col-lg-3")
      col_lg_five = stat_row.find("div", class_="col-lg-5")

      # Extract player details (Name, Position, Team)
      if col_lg_three:
          spans = col_lg_three.find_all("span", limit=2)
          player_name = spans[0].text.strip() if len(spans) > 0 else None
          span2 = spans[1].text.strip() if len(spans) > 1 else None
          player_position, player_team = span2.split(" ") if span2 else (None, None)

      if col_lg_five:
          spans = col_lg_five.find_all("span", limit=8)
          player_year = spans[1].text.strip() if len(spans) > 1 else None
          player_height = spans[3].text.strip() if len(spans) > 3 else None
          player_weight = spans[5].text.strip() if len(spans) > 5 else None

      # Extract injury details in a separate loop
      injury_table = t_row.find("table", class_="table-striped")
      if injury_table:
          tbody = injury_table.find("tbody")
          if tbody:
              injury_rows = tbody.find_all("tr")

              for injury_row in injury_rows:
                  columns = injury_row.find_all("td")
                  injury_date = columns[0].text.strip() if len(columns) > 0 else None
                  injury_type = columns[2].text.strip() if len(columns) > 2 else None
                  injury_details = columns[3].text.strip() if len(columns) > 3 else None
                  inj_specs = injury_type.split()
                  for i, inj_spec in enumerate(inj_specs):


                # Add extracted data to DataFrame
                  new_data =pd.DataFrame({
                    'Name': [player_name],
                    'Seasons Played': [player_year],
                    'Height': [player_height],
                    'Weight': [player_weight],
                    'Position': [player_position],
                    'Injury Date': [injury_date],
                    'Injury Type': [injury_type],
                    'Injury Details': [injury_details]
                    })
                  players_data = pd.concat([players_data, new_data], ignore_index=True)
# Save collected data to a CSV file
csv_filename = "players_data_qb.csv"
players_data.to_csv(csv_filename, index=False)

print(f"Data successfully saved to {csv_filename}")

Injury Specific 0: Shoulder
Injury Specific 1: Clavicle
Injury Specific 2: Fracture
Injury Specific 0: Hand
Injury Specific 1: Thumb
Injury Specific 2: Fracture
Injury Specific 0: Pedal
Injury Specific 1: Achilles
Injury Specific 2: Tear
Injury Specific 3: Grade
Injury Specific 4: 3
Injury Specific 0: Pedal
Injury Specific 1: Achilles
Injury Specific 2: Tear
Injury Specific 3: Grade
Injury Specific 4: 3
Injury Specific 0: Shoulder
Injury Specific 1: Scapula
Injury Specific 2: Fracture
Injury Specific 0: Shoulder
Injury Specific 1: Rotator
Injury Specific 2: Cuff
Injury Specific 3: Tear
Injury Specific 0: Back
Injury Specific 1: Lower
Injury Specific 2: Lumbar
Injury Specific 3: Sprain
Injury Specific 0: Knee
Injury Specific 1: ACL
Injury Specific 2: Tear
Injury Specific 3: Grade
Injury Specific 4: 3
Injury Specific 0: Knee
Injury Specific 1: ACL
Injury Specific 2: Tear
Injury Specific 3: Grade
Injury Specific 4: 3
Injury Specific 0: Knee
Injury Specific 1: LCL
Injury Specific 2: Sprain

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder


players_df = pd.read_csv('players_data.csv')

players_df['Position'].unique()

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False).set_output(transform='pandas')

ohetransform = ohe.fit_transform(players_df[['Position']])


players_df_encoded = pd.concat([players_df, ohetransform], axis=1).drop(columns=['Position'])

players_df_encoded.head(10)






,Name,Seasons Played,Height,Weight,Injury Date,Injury Type,Injury Details,Position_CB,Position_DB,Position_DE,Position_DT,Position_LB,Position_QB,Position_RB,Position_S,Position_TE,Position_WR
0,Trey Lance,4,"6'4""",226,"Sep 18, 2022",Pedal Ankle Dislocation,Lance's injury came on a zone-read run.,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,Trey Lance,4,"6'4""",226,"Sep 18, 2022",Pedal Ankle Fracture,Lance's injury came on a zone-read run.,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,Trey Lance,4,"6'4""",226,"Oct 10, 2021",Knee Strain Grade 1,Lance suffered a left knee sprain in the 49ers...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,Trey Lance,4,"6'4""",226,"Aug 29, 2021",Hand Finger Fracture,Lance suffered a finger injury during the pres...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,Trey Lance,4,"6'4""",226,"Feb 15, 2020",Knee Meniscus Trim,Trey Lance used crutches following a medical p...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5,Mitchell Trubisky,8,"6'2""",222,"Sep 29, 2019",Shoulder Labrum Tear,Injured his left shoulder on the first drive o...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6,Mitchell Trubisky,8,"6'2""",222,"Nov 18, 2018",Shoulder,Trubisky missed 2 games with a shoulder bruise.,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
7,Kenny Pickett,4,"6'3""",220,"Dec 29, 2024",Chest Rib Fracture,Pickett was forced to leave in the Week 17 con...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
8,Kenny Pickett,4,"6'3""",220,"Dec 3, 2023",Pedal Ankle (high) Sprain Grade 3,Pickett suffered a high ankle injury late in t...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
9,Kenny Pickett,4,"6'3""",220,"Oct 29, 2023",Chest Rib Bruise,Pickett sustained a rib injury near the end of...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [ ]:
#HotEncodingfor injury data

#import pandas as pd
#from sklearn.preprocessing import OneHotEncoder


#players_df = pd.read_csv('players_data.csv')

#players_df['Injury Type'].unique()




In [ ]:
#Peomt learning
from pyspark.sql import SparkSession
from pyspark.sql.functions import format_string

# Initialize Spark Session
spark = SparkSession.builder.appName("TabularToSentences").getOrCreate()

# Sample Data
data = [("Alice", 30, "Paris"), ("Bob", 25, "London")]
columns = ["name", "age", "city"]
df = spark.createDataFrame(data, columns)

# Define Sentence Structure
df = df.withColumn(
    "sentence",
    format_string("%s is %d years old and lives in %s", "name", "age", "city")
)

df.show(truncate=False)

+-----+---+------+----------------------------------------+
|name |age|city  |sentence                                |
+-----+---+------+----------------------------------------+
|Alice|30 |Paris |Alice is 30 years old and lives in Paris|
|Bob  |25 |London|Bob is 25 years old and lives in London |
+-----+---+------+----------------------------------------+



In [ ]:
'''

#Propmt learning
import pyspark
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import format_string

# Initialize Spark Session
spark = SparkSession.builder.appName("TabularToSentences").getOrCreate()

# gather data
Injury_df = pd.read_csv('players_data.csv')
columns = ["Name", "Seasons Played", "Height", "Weight", "Position", "Injury Date", "Injury Type", "Injury Details"]
df = spark.createDataFrame(Injury_df, columns)

df = df.withColumn(
    "sentence",
    format_string("%s is %d , %d lbs, %s. He has played %d seasons. Key Injury - %s, (%s) ","Name","Height", "Weight","Position","Seasons Played","Injury Type","Injury Details")


)




In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws, col, lit

def tabular_to_sentences(input_path, output_path, sentence_template):
    """
    Convert tabular data to structured sentences using PySpark

    Parameters:
    input_path (str): Path to input CSV file
    output_path (str): Path to output CSV file
    sentence_template (dict): Dictionary mapping column names to their description in the sentence
        Example: {'name': 'The person named', 'age': 'is', 'city': 'lives in'}
    """
    # Initialize Spark session
    spark = SparkSession.builder \
        .appName("Tabular to Sentences") \
        .getOrCreate()

    # Read input CSV
    df = spark.read.csv(input_path, header=True, inferSchema=True)

    # Create the sentence by concatenating columns with their descriptions
    sentence_parts = []
    for col_name, description in sentence_template.items():
        sentence_parts.extend([lit(description), col(col_name)])

    # Generate complete sentences
    df_with_sentences = df.withColumn(
        "generated_sentence",
        concat_ws(" ", *sentence_parts)
    )

    # Save to CSV
    df_with_sentences.toPandas().to_csv(output_path, index=False)

    # Stop Spark session
    spark.stop()

# Example usage
if __name__ == "__main__":
    # Sample data structure and template
    template = {
        'name': 'The person named',
        'age': 'is',
        'years_old': 'years old and',
        'city': 'lives in'
    }

    # Example paths
    input_file = "input_data.csv"
    output_file = "output_with_sentences.csv"

    tabular_to_sentences(input_file, output_file, template)

# Sample input CSV structure:
"""
name,age,years_old,city
John,25,25,New York
Alice,30,30,San Francisco
"""

# Generated output would look like:
"""
name,age,years_old,city,generated_sentence
John,25,25,New York,The person named John is 25 years old and lives in New York
Alice,30,30,San Francisco,The person named Alice is 30 years old and lives in San Francisco
"""

In [ ]:
Sentence_Template = {
    "name": "",
    "Height": 0.0,
    "Weight": 0,
    "Position": "",
    "Seasons Played": 0,
    "Injury Type": "",
    "Injury Details": ""

}

In [ ]:
#Make structred sentence template
Sentence_Template = {
    "name": "",
    "Height":'has a height of',
    "Weight":'his weight is' ,
    "Position": "pounds and he plays the position of ",
    "Seasons Played":'. He has played ' ,
    "Injury Details": "Seasons in the NFL.",


}


#Import Libraries
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws, col, lit

#Initialize Spark Environment
spark = SparkSession.builder \
  .appName("Player Analysis") \
  .getOrCreate()
#Read csv file data
df = spark.read.csv("players_data.csv", header=True, inferSchema=True)

#make sentence loop
sentence_parts = []
for col_name, description in Sentence_Template.items():
    sentence_parts.extend([lit(description), col(col_name)])

#
generated_sentence_df = df.withColumn(
    "generated_sentence",
    concat_ws(" ", *sentence_parts)
)
generated_sentence_df.show(truncate=False)

generated_sentence_df.toPandas().to_csv("output_with_sentences.csv", index=False)

spark.stop()






+-----------------+--------------+------+------+--------+------------+----------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Name             |Seasons Played|Height|Weight|Position|Injury Date |Injury Type                       |Injury Details                                                                                                                                        |generated_sentence                                                                                                                                                                                    